In [1]:
!pip install z3-solver
from z3 import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 44.2 MB/s eta 0:00:00


In [2]:
# kada napisemo x = Int('x') ne pravimo obicnu promenljivu sa konkretnom vr.,
# vec promenljivu nad kojom z3 rezonuje

In [3]:
x = Int('x')
y = Int('y')

s = Solver() # cuva ogranicenja

s.add(x > 0) # dodaje nove uslove
s.add(y == x + 2)

print(s.check()) # proverava da li postoji model koji zadovoljava zadate uslove -> sat, unsat, unknown
print(s.model()) # daje jedan konkretan svet u kome su zadati uslovi tacni

sat
[y = 3, x = 1]


In [4]:
x = Int('x')
y = Int('y')

s = Solver()

s.add(x + y == 10)
s.add(x > y)

print(s.check())

m = s.model()
print(m)

print("x  = ", m.evaluate(x)) # izracunava vrednost logickih ili matematickih izraza u kontekstu resenja
print("y  = ", m.evaluate(y))
print("x + y = ", m.evaluate(x + y))

sat
[y = 4, x = 6]
x  =  6
y  =  4
x + y =  10


In [5]:
Osoba = DeclareSort('Osoba') # uvodi domen novog objekta

Sokrat = Const('Sokrat', Osoba) # uvodi jedan konkretan objekat iz zadatog domena

Covek = Function('Covek', Osoba, BoolSort()) # uvodi predikat Covek: Osoba -> Bool, predikat vraca bool
Smrtan = Function('Smrtan', Osoba, BoolSort())

Otac = Function('Otac', Osoba, Osoba) # uvodi funkciju Otac: Osoba -> Osoba, fja vraca objekat

x = Const('x', Osoba) # omogucava nam da pisem opsta tvrdjenja

In [6]:
# premise
# 1. svi ljudi su smrtni
# 2. sokrat je covek

# zakljucak
# 3. sokrat je smrtan

s = Solver()
s.add(ForAll(x, Implies(Covek(x), Smrtan(x))))
s.add(Covek(Sokrat))
# dodajemo negaciju zakljucka kako bismo proverili da li vazi ili ne
s.add(Not(Smrtan(Sokrat)))

print(s.check())

unsat


In [7]:
# premise
# 1. svi studenti koji slusaju vi znaju pajton
# 2. pera zna pajton
# zakljucak
# 3. pera slusa vi

Osoba = DeclareSort('Osoba')

SlusaVI = Function('SlusaVi', Osoba, BoolSort())
ZnaPajton = Function('ZnaPajton', Osoba, BoolSort())

Pera = Const('Pera', Osoba)

s = Solver()
s.add(ForAll(x, Implies(SlusaVI(x), ZnaPajton(x))))
s.add(ZnaPajton(Pera))
s.add(Not(SlusaVI(Pera)))

print(s.check())
print(s.model())

sat
[Pera = Osoba!val!0,
 ZnaPajton = [else -> True],
 SlusaVi = [else -> False]]


In [8]:
# premise
# 1. svi studenti koji slusaju vi rade domaci
# 2. pera slusa vi
# zakljucak
# 3. pera radi domaci

Osoba = DeclareSort('Osoba')

SlusaVi = Function('SlusaVi', Osoba, BoolSort())
RadiDomaci = Function('RadiDomaci', Osoba, BoolSort())

Pera = Const('Pera', Osoba)

s = Solver()
s.add(ForAll(x, Implies(SlusaVI(x), RadiDomaci(x))))
s.add(SlusaVI(Pera))
s.add(Not(RadiDomaci(Pera)))

print(s.check())

unsat


In [9]:
# predikati mogu imati vise argumenata

# premise
# 1. ako je x roditelj y, onda je x predak y
# 2. ako je x roditelj y i y roditelj z, onda je x predak z
# 3. Marko je roditelj Ane
# 4. Ana je roditelj Jovana
# zakljucak
# 5. Marko je predak Jovana

Osoba = DeclareSort('Osoba')

Roditelj = Function('Roditelj', Osoba, Osoba, BoolSort())
Predak = Function('Predak', Osoba, Osoba, BoolSort())

Marko = Const('Marko', Osoba)
Ana = Const('Ana', Osoba)
Jovan = Const('Jovan', Osoba)

x, y, z = Consts('x y z', Osoba)

s = Solver()
s.add(ForAll([x, y], Implies(Roditelj(x, y), Predak(x, y))))
s.add(ForAll([x, y, z], Implies(And(Roditelj(x, y), Roditelj(y, z)), Predak(x, z))))
s.add(Roditelj(Marko, Ana))
s.add(Roditelj(Ana, Jovan))
s.add(Not(Predak(Marko, Jovan)))

print(s.check())

unsat


In [10]:
# premise
# 1. Ako su dve osobe braca, imaju zajedničkog roditelja.
# 2. Roditelj je stariji od deteta.
# 3. Postoje dve osobe koje su braća.
# zakljucak
# Postoji neko ko je stariji od nekoga.

Osoba = DeclareSort('Osoba')

Braca = Function('Braca', Osoba, Osoba, BoolSort())
Roditelj = Function('Roditelj', Osoba, Osoba, BoolSort())
Stariji = Function('Stariji', Osoba, Osoba, BoolSort())

s = Solver()
s.add(ForAll([x, y], Implies(Braca(x, y), Exists(z, And(Roditelj(z, x), Roditelj(z, y))))))
s.add(ForAll([x, y], Implies(Roditelj(x, y), Stariji(x, y))))
s.add(Exists([x, y], Braca(x, y)))
s.add(Not(Exists([x, y], Stariji(x, y))))

print(s.check())

unsat


In [12]:
# premise:
# 1. Ako su dve prave nemimoilazne, onda se seku ili su paralelne.
# 2. Ako se dve prave seku, onda su u istoj ravni.
# 3. Ako su dve prave paralelne, onda su u istoj ravni.
# zaključak:
# Ako su dve prave nemimoilazne, onda su u istoj ravni.

Prava = DeclareSort('Prava')

Nemimoilazne = Function('Nemimoilazne', Prava, Prava, BoolSort())
SekuSe = Function('SekuSe', Prava, Prava, BoolSort())
Paralelne = Function('Paralelne', Prava, Prava, BoolSort())
UIstojRavni = Function('UIstojRavni', Prava, Prava, BoolSort())

x, y = Consts('x y', Prava)

s = Solver()

axioms = [
    ForAll([x, y], Implies(Nemimoilazne(x, y), Or(SekuSe(x, y), Paralelne(x, y)))),
    ForAll([x, y], Implies(SekuSe(x, y), UIstojRavni(x, y))),
    ForAll([x, y], Implies(Paralelne(x, y), UIstojRavni(x, y)))
]
s.add(axioms)

conjecture = ForAll([x, y], Implies(Nemimoilazne(x, y), UIstojRavni(x, y)))
s.add(Not(conjecture))

print(s.check())

unsat


In [13]:
# premise:
# 1. Svaki grad pripada nekoj državi.
# 2. Ako grad pripada državi, onda je ta država povezana sa tim gradom.
# 3. Beograd je grad.
# zaključak:
# Postoji država koja je povezana sa Beogradom.

Grad = DeclareSort('Grad')
Drzava = DeclareSort('Drzava')

Pripada = Function('Pripada', Grad, Drzava, BoolSort())
Povezana = Function('Povezana', Drzava, Grad, BoolSort())

beograd = Const('beograd', Grad)
x = Const('x', Grad)

y = Const('y', Drzava)

s = Solver()

axioms = [
    ForAll(x, Exists(y, Pripada(x, y))),
    ForAll([x, y], Implies(Pripada(x, y), Povezana(y, x))),
]
s.add(axioms)

conjecture = Exists(y, Povezana(y, beograd))
s.add(Not(conjecture))

print(s.check())

unsat


In [15]:
# premise:
# 1. Svaki grad pripada nekoj državi.
# 2. Beograd je grad.
# pogrešan zaključak:
# Beograd pripada Srbiji.

Grad = DeclareSort('Grad')
Drzava = DeclareSort('Drzava')

Pripada = Function('Pripada', Grad, Drzava, BoolSort())

beograd = Const('beograd', Grad)
x = Const('x', Grad)

srbija = Const('srbija', Drzava)
y = Const('y', Drzava)

s = Solver()

axioms = [
    ForAll(x, Exists(y, Pripada(x, y))),
]
s.add(axioms)

conjecture = Pripada(beograd, srbija)
s.add(Not(conjecture))

print(s.check())
print(s.model())

sat
[beograd = Grad!val!0,
 srbija = Drzava!val!0,
 Pripada = [else -> Var(1) == Drzava!val!1]]


In [16]:
# Možemo izraziti da je funkcija injektivna.
# Za funkciju: f: A -> A
# injektivnost znači: f(x) = f(y) => x = y

A = DeclareSort('A')

f = Function('f', A, A)

x, y = Consts('x y', A)
a, b = Consts('a b', A)

s = Solver()
s.add(ForAll([x, y], Implies(f(x) == f(y), x == y)))
s.add(a != b)
s.add(f(a) == f(b))
print(s.check())

unsat
